In [2]:
import json
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

JSONL_PATH = "/home/as5606/projects/LLaVA-Med/data/chest_xray_filter_decisions.jsonl"

entries = [json.loads(l) for l in open(JSONL_PATH)]

# strip <image> tags from questions
def clean(q):
    return re.sub(r'<image>\s*|\s*<image>', '', q).strip()

questions = [clean(e["question"]) for e in entries]
labels    = [1 if e["decision"] == "KEEP" else 0 for e in entries]

print(f"Total: {len(questions)}  |  KEEP: {sum(labels)}  |  REMOVE: {len(labels)-sum(labels)}")


Total: 20886  |  KEEP: 2143  |  REMOVE: 18743


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    questions, labels, test_size=0.2, random_state=42, stratify=labels
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

clf = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0)
clf.fit(X_train_vec, y_train)

threshold = 0.65  # tune this — higher = stricter KEEP
y_prob = clf.predict_proba(X_test_vec)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

print(classification_report(y_test, y_pred, target_names=["REMOVE", "KEEP"]))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

      REMOVE       0.98      0.93      0.96      3749
        KEEP       0.59      0.86      0.70       429

    accuracy                           0.92      4178
   macro avg       0.78      0.89      0.83      4178
weighted avg       0.94      0.92      0.93      4178

Confusion matrix (rows=actual, cols=predicted):
[[3488  261]
 [  61  368]]


In [4]:
feature_names = vectorizer.get_feature_names_out()
coefs = clf.coef_[0]

top_keep   = np.argsort(coefs)[-15:][::-1]
top_remove = np.argsort(coefs)[:15]

print("Top words → KEEP:")
for i in top_keep:
    print(f"  {feature_names[i]:40s}  {coefs[i]:+.3f}")

print("\nTop words → REMOVE:")
for i in top_remove:
    print(f"  {feature_names[i]:40s}  {coefs[i]:+.3f}")


Top words → KEEP:
  where                                     +5.151
  position                                  +4.787
  show about                                +4.632
  arrow indicate                            +4.033
  arrow                                     +3.978
  the left                                  +3.854
  which                                     +3.763
  the position                              +3.636
  located                                   +3.580
  where is                                  +3.570
  position of                               +3.356
  the right                                 +3.315
  in                                        +3.289
  affected                                  +3.266
  the                                       +3.259

Top words → REMOVE:
  what                                      -4.046
  mean                                      -3.946
  is there                                  -3.644
  purpose                                  

In [5]:
def predict(question):
    q = clean(question)
    vec = vectorizer.transform([q])
    pred = clf.predict(vec)[0]
    prob = clf.predict_proba(vec)[0]
    label = "KEEP" if pred == 1 else "REMOVE"
    return label, prob[pred]

# test it
print(predict("What is the appearance of the right lower zone?"))
print(predict("What does ground-glass opacity mean?"))
print(predict("What is the patient's age?"))


('KEEP', np.float64(0.983076324961213))
('REMOVE', np.float64(0.9541810949327553))
('REMOVE', np.float64(0.9817528682111514))


In [7]:
import json
import re
import numpy as np
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix

JSONL_PATH = "/home/as5606/projects/LLaVA-Med/data/chest_xray_filter_decisions.jsonl"

def clean(q):
    return re.sub(r'<image>\s*|\s*<image>', '', q).strip()

entries = [json.loads(l) for l in open(JSONL_PATH)]
all_questions = [clean(e["question"]) for e in entries]
all_labels    = [1 if e["decision"] == "KEEP" else 0 for e in entries]

print(f"Total: {len(all_questions)}  |  KEEP: {sum(all_labels)}  |  REMOVE: {len(all_labels)-sum(all_labels)}")


Total: 20886  |  KEEP: 2143  |  REMOVE: 18743


In [8]:
import json
import torch
import os
from PIL import Image
from open_clip import create_model_from_pretrained, get_tokenizer
from tqdm import tqdm


DEVICE = "cuda"
model, preprocess = create_model_from_pretrained(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224',
    device=DEVICE
)
tokenizer = get_tokenizer(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
model.eval()

/home/as5606/miniconda3/envs/biomedclip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CustomTextCLIP(
  (visual): TimmModel(
    (trunk): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
          

In [9]:
BATCH = 128

embeddings = []
model.eval()
with torch.no_grad():
    for i in tqdm(range(0, len(all_questions), BATCH), desc="Encoding questions"):
        batch = all_questions[i : i + BATCH]
        tokens = tokenizer(batch).to(DEVICE)
        feats = model.encode_text(tokens)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        embeddings.append(feats.cpu())

X = torch.cat(embeddings).numpy()  # [N, 512]
y = np.array(all_labels)

print(f"Embedding matrix: {X.shape}")



Encoding questions: 100%|████████████████████████████████████████████████████| 164/164 [01:09<00:00,  2.37it/s]

Embedding matrix: (20886, 512)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# compute per-sample weights to handle 9:1 imbalance
sample_weights = np.where(y_train == 1, 9.0, 1.0)

clf = MLPClassifier(
    hidden_layer_sizes=(256, 64),
    activation='relu',
    max_iter=200,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    verbose=True
)
clf.fit(X_train, y_train, sample_weight=sample_weights)


Iteration 1, loss = 0.45239766
Validation score: 0.842053
Iteration 2, loss = 0.33081191
Validation score: 0.856203
Iteration 3, loss = 0.30561620
Validation score: 0.862455
Iteration 4, loss = 0.28574960
Validation score: 0.873643
Iteration 5, loss = 0.26470520
Validation score: 0.878908
Iteration 6, loss = 0.24724746
Validation score: 0.873314
Iteration 7, loss = 0.23551773
Validation score: 0.875946
Iteration 8, loss = 0.22112924
Validation score: 0.880882
Iteration 9, loss = 0.20463685
Validation score: 0.877262
Iteration 10, loss = 0.19512393
Validation score: 0.879566
Iteration 11, loss = 0.18890637
Validation score: 0.886147
Iteration 12, loss = 0.17352129
Validation score: 0.863771
Iteration 13, loss = 0.16279729
Validation score: 0.857848
Iteration 14, loss = 0.15902101
Validation score: 0.859493
Iteration 15, loss = 0.15583974
Validation score: 0.883185
Iteration 16, loss = 0.14395608
Validation score: 0.868707
Iteration 17, loss = 0.13906407
Validation score: 0.876933
Iterat

,hidden_layer_sizes,"(256, ...)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,200
,shuffle,True
,random_state,42


In [12]:
threshold = 0.65  # tune if needed

y_prob = clf.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

print(classification_report(y_test, y_pred, target_names=["REMOVE", "KEEP"]))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

      REMOVE       0.99      0.89      0.94      3749
        KEEP       0.50      0.91      0.64       429

    accuracy                           0.90      4178
   macro avg       0.74      0.90      0.79      4178
weighted avg       0.94      0.90      0.91      4178

Confusion matrix (rows=actual, cols=predicted):
[[3353  396]
 [  38  391]]


In [13]:
!nvidia-smi


Wed Jul 22 18:18:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.45.04              Driver Version: 595.45.04      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          On  |   00000000:1B:00.0 Off |                    0 |
| N/A   42C    P0             34W /  250W |    2095MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

JSONL_PATH = "/home/as5606/projects/LLaVA-Med/data/chest_xray_filter_decisions.jsonl"
MODEL_NAME  = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

DEVICE      = "cuda"
EPOCHS      = 8
BATCH_SIZE  = 32
MAX_LEN     = 64
THRESHOLD   = 0.65

entries    = [json.loads(l) for l in open(JSONL_PATH)]
statements = [e["statement"].strip() for e in entries]
labels     = [1 if e["decision"] == "KEEP" else 0 for e in entries]

X_train, X_test, y_train, y_test = train_test_split(
    statements, labels, test_size=0.2, stratify=labels, random_state=42
)
print(f"Train: {len(X_train)}  Test: {len(X_test)}")


/home/as5606/miniconda3/envs/biomedclip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 1714  Test: 429


In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class QDataset(Dataset):
    def __init__(self, texts, labels):
        enc = tokenizer(texts, truncation=True, padding=True,
                        max_length=MAX_LEN, return_tensors="pt")
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels         = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx]
        }

train_ds = QDataset(X_train, y_train)
test_ds  = QDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)


In [3]:
model_bert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(DEVICE)

# 9:1 imbalance → weight KEEP class 9x higher
class_weights = torch.tensor([1.0, 9.0]).to(DEVICE)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

optimizer = AdamW(model_bert.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)


Loading weights: 100%|████████████████████████████████████████████████████| 199/199 [00:00<00:00, 20789.74it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING   

In [4]:
for epoch in range(EPOCHS):
    model_bert.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels_b       = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels_b)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model_bert.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} — avg loss: {total_loss/len(train_loader):.4f}")


Epoch 1/8: 100%|███████████████████████████████████████████████████████████████| 54/54 [00:03<00:00, 14.60it/s]


Epoch 1 — avg loss: 0.1027


Epoch 2/8: 100%|███████████████████████████████████████████████████████████████| 54/54 [00:02<00:00, 23.63it/s]


Epoch 2 — avg loss: 0.0007


Epoch 3/8: 100%|███████████████████████████████████████████████████████████████| 54/54 [00:02<00:00, 23.74it/s]


...

Epoch 6 — avg loss: 0.0002


Epoch 7/8: 100%|███████████████████████████████████████████████████████████████| 54/54 [00:02<00:00, 23.63it/s]


Epoch 7 — avg loss: 0.0001


Epoch 8/8: 100%|███████████████████████████████████████████████████████████████| 54/54 [00:02<00:00, 23.64it/s]

Epoch 8 — avg loss: 0.0001


In [5]:
model_bert.eval()
all_probs = []
all_labels_eval = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels_eval.extend(batch["labels"].numpy())

y_pred = (np.array(all_probs) >= THRESHOLD).astype(int)
print(classification_report(all_labels_eval, y_pred, target_names=["REMOVE", "KEEP"]))
print("Confusion matrix:")
print(confusion_matrix(all_labels_eval, y_pred))


Evaluating: 100%|██████████████████████████████████████████████████████████████| 14/14 [00:00<00:00, 54.07it/s]


ValueError: Number of classes, 1, does not match size of target_names, 2. Try specifying the labels parameter

In [1]:
import json
import re
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

JSONL_PATH = "/home/as5606/projects/Super_LlaVA_Med/chest_xray_all_statements.jsonl"
MODEL_NAME  = "distilbert-base-uncased"
DEVICE      = "cuda"
EPOCHS      = 5
BATCH_SIZE  = 32
MAX_LEN     = 64
THRESHOLD   = 0.65

def clean(q):
    return re.sub(r'<image>\s*|\s*<image>', '', q).strip()

entries   = [json.loads(l) for l in open(JSONL_PATH)]
questions = [clean(e["statement"]) for e in entries]
labels    = [1 if e["decision"] == "KEEP" else 0 for e in entries]

X_train, X_test, y_train, y_test = train_test_split(
    questions, labels, test_size=0.2, stratify=labels, random_state=42
)
print(f"Train: {len(X_train)}  Test: {len(X_test)}")


/home/as5606/miniconda3/envs/biomedclip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 16708  Test: 4178


In [2]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class QDataset(Dataset):
    def __init__(self, texts, labels):
        enc = tokenizer(texts, truncation=True, padding=True,
                        max_length=MAX_LEN, return_tensors="pt")
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels         = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx]
        }

train_ds = QDataset(X_train, y_train)
test_ds  = QDataset(X_test,  y_test)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)


In [3]:
model_bert = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, use_safetensors=True
).to(DEVICE)

class_weights = torch.tensor([1.0, 9.0]).to(DEVICE)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

optimizer = AdamW(model_bert.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)



Loading weights: 100%|█████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 168.07it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
for epoch in range(EPOCHS):
    model_bert.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels_b       = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        loss = loss_fn(model_bert(input_ids=input_ids, attention_mask=attention_mask).logits, labels_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_bert.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} — avg loss: {total_loss/len(train_loader):.4f}")


Epoch 1/5: 100%|████████████████████████████████████████████████████████████████████████████| 523/523 [00:24<00:00, 21.36it/s]


Epoch 1 — avg loss: 0.5171


Epoch 2/5: 100%|████████████████████████████████████████████████████████████████████████████| 523/523 [00:22<00:00, 22.81it/s]


Epoch 2 — avg loss: 0.3977


Epoch 3/5: 100%|████████████████████████████████████████████████████████████████████████████| 523/523 [00:23<00:00, 22.59it/s]


Epoch 3 — avg loss: 0.3484


Epoch 4/5: 100%|████████████████████████████████████████████████████████████████████████████| 523/523 [00:23<00:00, 22.49it/s]


Epoch 4 — avg loss: 0.3126


Epoch 5/5: 100%|████████████████████████████████████████████████████████████████████████████| 523/523 [00:23<00:00, 22.41it/s]

Epoch 5 — avg loss: 0.2823


In [5]:
model_bert.eval()
all_probs, all_true = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        probs = torch.softmax(
            model_bert(input_ids=batch["input_ids"].to(DEVICE),
                       attention_mask=batch["attention_mask"].to(DEVICE)).logits,
            dim=-1
        )[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_true.extend(batch["labels"].numpy())

y_pred = (np.array(all_probs) >= THRESHOLD).astype(int)
print(classification_report(all_true, y_pred, target_names=["REMOVE", "KEEP"]))
print(confusion_matrix(all_true, y_pred))


Evaluating: 100%|███████████████████████████████████████████████████████████████████████████| 131/131 [00:01<00:00, 81.75it/s]

              precision    recall  f1-score   support

      REMOVE       0.96      0.91      0.94      3749
        KEEP       0.48      0.70      0.57       429

    accuracy                           0.89      4178
   macro avg       0.72      0.81      0.76      4178
weighted avg       0.91      0.89      0.90      4178

[[3427  322]
 [ 128  301]]
